In [1]:
import pandas as pd
import numpy as np 

### Carga de datos

In [2]:
df = pd.read_csv("../data/raw/social_media_vs_productivity.csv")

# Información básica del dataset

In [3]:
print("=== INFORMACIÓN BÁSICA ===")
print(f"Forma del dataset: {df.shape}")
print(f"\nTipos de datos:")
print(df.dtypes)

print("\n=== PRIMERAS 5 FILAS ===")
print(df.head())

print("\n=== INFORMACIÓN ESTADÍSTICA ===")
print(df.describe())

print("\n=== VALORES NULOS ===")
print(df.isnull().sum())

print("\n=== VALORES ÚNICOS POR COLUMNA ===")
for col in df.columns:
    if df[col].dtype == 'object':
        print(f"{col}: {df[col].nunique()} valores únicos - {df[col].unique()[:10]}")
    else:
        print(f"{col}: {df[col].nunique()} valores únicos")

=== INFORMACIÓN BÁSICA ===
Forma del dataset: (30000, 19)

Tipos de datos:
age                                 int64
gender                             object
job_type                           object
daily_social_media_time           float64
social_platform_preference         object
number_of_notifications             int64
work_hours_per_day                float64
perceived_productivity_score      float64
actual_productivity_score         float64
stress_level                      float64
sleep_hours                       float64
screen_time_before_sleep          float64
breaks_during_work                  int64
uses_focus_apps                      bool
has_digital_wellbeing_enabled        bool
coffee_consumption_per_day          int64
days_feeling_burnout_per_month      int64
weekly_offline_hours              float64
job_satisfaction_score            float64
dtype: object

=== PRIMERAS 5 FILAS ===
   age  gender    job_type  daily_social_media_time  \
0   56    Male  Unemployed      

# Análisis más detallado de los problemas

In [4]:
print("=== ANÁLISIS DE PROBLEMAS DETECTADOS ===")

# 1. Porcentaje de valores nulos por columna
print("\n1. PORCENTAJE DE VALORES NULOS:")
null_percentages = (df.isnull().sum() / len(df)) * 100
for col, pct in null_percentages.items():
    if pct > 0:
        print(f"   {col}: {pct:.2f}%")

# 2. Verificar outliers en variables numéricas
print("\n2. POSIBLES OUTLIERS (valores extremos):")
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if col != 'age':  # La edad es normal que tenga rango amplio
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)][col].count()
        if outliers > 0:
            print(f"   {col}: {outliers} outliers ({(outliers/len(df))*100:.2f}%)")

# 3. Verificar consistencia en datos categóricos
print("\n3. DATOS CATEGÓRICOS:")
print(f"   Gender: {df['gender'].value_counts().to_dict()}")
print(f"   Job types: {df['job_type'].value_counts().to_dict()}")
print(f"   Social platforms: {df['social_platform_preference'].value_counts().to_dict()}")

# 4. Verificar rangos lógicos
print("\n4. VERIFICACIÓN DE RANGOS LÓGICOS:")
print(f"   Horas de trabajo por día - Min: {df['work_hours_per_day'].min():.2f}, Max: {df['work_hours_per_day'].max():.2f}")
print(f"   Horas de sueño - Min: {df['sleep_hours'].min():.2f}, Max: {df['sleep_hours'].max():.2f}")
print(f"   Nivel de estrés - Min: {df['stress_level'].min():.2f}, Max: {df['stress_level'].max():.2f}")
print(f"   Tiempo redes sociales - Min: {df['daily_social_media_time'].min():.2f}, Max: {df['daily_social_media_time'].max():.2f}")

=== ANÁLISIS DE PROBLEMAS DETECTADOS ===

1. PORCENTAJE DE VALORES NULOS:
   daily_social_media_time: 9.22%
   perceived_productivity_score: 5.38%
   actual_productivity_score: 7.88%
   stress_level: 6.35%
   sleep_hours: 8.66%
   screen_time_before_sleep: 7.37%
   job_satisfaction_score: 9.10%

2. POSIBLES OUTLIERS (valores extremos):
   daily_social_media_time: 226 outliers (0.75%)
   number_of_notifications: 261 outliers (0.87%)
   work_hours_per_day: 97 outliers (0.32%)
   screen_time_before_sleep: 98 outliers (0.33%)
   coffee_consumption_per_day: 127 outliers (0.42%)
   weekly_offline_hours: 116 outliers (0.39%)

3. DATOS CATEGÓRICOS:
   Gender: {'Male': 14452, 'Female': 14370, 'Other': 1178}
   Job types: {'Education': 5055, 'IT': 5026, 'Finance': 5017, 'Student': 5012, 'Unemployed': 4958, 'Health': 4932}
   Social platforms: {'TikTok': 6096, 'Telegram': 6013, 'Instagram': 6006, 'Twitter': 5964, 'Facebook': 5921}

4. VERIFICACIÓN DE RANGOS LÓGICOS:
   Horas de trabajo por día - 

In [ ]:
# LIMPIEZA COMPLETA DE DATOS

In [5]:
print("=== INICIANDO LIMPIEZA DE DATOS ===")

# Crear una copia para trabajar
df_clean = df.copy()
print(f"Dataset original: {df_clean.shape}")

# 1. TRATAMIENTO DE VALORES NULOS
print("\n1. TRATANDO VALORES NULOS...")

# Para variables numéricas, usar la mediana (más robusta que la media)
numeric_cols_with_nulls = ['daily_social_media_time', 'perceived_productivity_score', 
                          'actual_productivity_score', 'stress_level', 'sleep_hours', 
                          'screen_time_before_sleep', 'job_satisfaction_score']

for col in numeric_cols_with_nulls:
    median_val = df_clean[col].median()
    nulls_before = df_clean[col].isnull().sum()
    df_clean[col].fillna(median_val, inplace=True)
    print(f"   {col}: {nulls_before} nulos rellenados con mediana {median_val:.2f}")

# 2. TRATAMIENTO DE OUTLIERS
print("\n2. TRATANDO OUTLIERS...")

def remove_outliers_iqr(df, column, factor=1.5):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - factor * IQR
    upper_bound = Q3 + factor * IQR
    
    outliers_mask = (df[column] < lower_bound) | (df[column] > upper_bound)
    outliers_count = outliers_mask.sum()
    
    # En lugar de eliminar, vamos a "winsorizar" (limitar a los percentiles)
    df[column] = np.where(df[column] < lower_bound, lower_bound, df[column])
    df[column] = np.where(df[column] > upper_bound, upper_bound, df[column])
    
    return outliers_count

# Aplicar tratamiento de outliers a columnas específicas
outlier_cols = ['daily_social_media_time', 'work_hours_per_day', 'screen_time_before_sleep', 
                'coffee_consumption_per_day', 'weekly_offline_hours']

for col in outlier_cols:
    outliers_treated = remove_outliers_iqr(df_clean, col)
    print(f"   {col}: {outliers_treated} outliers tratados")

# 3. VALIDACIÓN DE RANGOS LÓGICOS
print("\n3. VALIDANDO RANGOS LÓGICOS...")

# Horas de trabajo: máximo 16 horas (muy extremo pero posible)
df_clean['work_hours_per_day'] = df_clean['work_hours_per_day'].clip(0, 16)

# Horas de sueño: entre 3 y 12 horas
df_clean['sleep_hours'] = df_clean['sleep_hours'].clip(3, 12)

# Nivel de estrés: entre 1 y 10
df_clean['stress_level'] = df_clean['stress_level'].clip(1, 10)

# Scores de productividad y satisfacción: entre 0 y 10
df_clean['perceived_productivity_score'] = df_clean['perceived_productivity_score'].clip(0, 10)
df_clean['actual_productivity_score'] = df_clean['actual_productivity_score'].clip(0, 10)
df_clean['job_satisfaction_score'] = df_clean['job_satisfaction_score'].clip(0, 10)

# Tiempo de pantalla antes de dormir: máximo 4 horas
df_clean['screen_time_before_sleep'] = df_clean['screen_time_before_sleep'].clip(0, 4)

# Consumo de café: máximo 8 tazas por día
df_clean['coffee_consumption_per_day'] = df_clean['coffee_consumption_per_day'].clip(0, 8)

print("   Rangos lógicos aplicados correctamente")

# 4. NORMALIZACIÓN DE DATOS CATEGÓRICOS
print("\n4. NORMALIZANDO DATOS CATEGÓRICOS...")

# Verificar si hay inconsistencias en mayúsculas/minúsculas
print(f"   Gender único: {df_clean['gender'].unique()}")
print(f"   Job types únicos: {df_clean['job_type'].unique()}")
print(f"   Platforms únicas: {df_clean['social_platform_preference'].unique()}")

print("\n=== RESUMEN POST-LIMPIEZA ===")
print(f"Forma final: {df_clean.shape}")
print(f"Valores nulos restantes: {df_clean.isnull().sum().sum()}")
print(f"Duplicados: {df_clean.duplicated().sum()}")

=== INICIANDO LIMPIEZA DE DATOS ===
Dataset original: (30000, 19)

1. TRATANDO VALORES NULOS...
   daily_social_media_time: 2765 nulos rellenados con mediana 3.03
   perceived_productivity_score: 1614 nulos rellenados con mediana 5.53
   actual_productivity_score: 2365 nulos rellenados con mediana 4.95
   stress_level: 1904 nulos rellenados con mediana 6.00
   sleep_hours: 2598 nulos rellenados con mediana 6.50
   screen_time_before_sleep: 2211 nulos rellenados con mediana 1.01
   job_satisfaction_score: 2730 nulos rellenados con mediana 4.95

2. TRATANDO OUTLIERS...
   daily_social_media_time: 348 outliers tratados
   work_hours_per_day: 97 outliers tratados
   screen_time_before_sleep: 198 outliers tratados
   coffee_consumption_per_day: 127 outliers tratados
   weekly_offline_hours: 116 outliers tratados

3. VALIDANDO RANGOS LÓGICOS...
   Rangos lógicos aplicados correctamente

4. NORMALIZANDO DATOS CATEGÓRICOS...
   Gender único: ['Male' 'Female' 'Other']
   Job types únicos: ['Une

C:\Users\anoni\AppData\Local\Temp\ipykernel_21512\1462069177.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean[col].fillna(median_val, inplace=True)
C:\Users\anoni\AppData\Local\Temp\ipykernel_21512\1462069177.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exa

Duplicados: 0


In [6]:
# 5. INGENIERÍA DE CARACTERÍSTICAS
print("=== INGENIERÍA DE CARACTERÍSTICAS ===")

# Crear nuevas variables derivadas que pueden ser útiles para ML
print("\n5. CREANDO NUEVAS CARACTERÍSTICAS...")

# Ratio de productividad (percibida vs real)
df_clean['productivity_gap'] = df_clean['perceived_productivity_score'] - df_clean['actual_productivity_score']

# Categoría de edad
df_clean['age_category'] = pd.cut(df_clean['age'], 
                                 bins=[17, 25, 35, 45, 55, 65], 
                                 labels=['18-25', '26-35', '36-45', '46-55', '56-65'])

# Categoría de uso de redes sociales
df_clean['social_media_usage'] = pd.cut(df_clean['daily_social_media_time'],
                                       bins=[0, 2, 4, 6, 20],
                                       labels=['Bajo', 'Medio', 'Alto', 'Muy Alto'])

# Categoría de horas de trabajo
df_clean['work_intensity'] = pd.cut(df_clean['work_hours_per_day'],
                                   bins=[0, 6, 8, 10, 16],
                                   labels=['Parcial', 'Normal', 'Intenso', 'Extremo'])

# Índice de bienestar digital (combinando varias métricas)
df_clean['digital_wellness_score'] = (
    (df_clean['uses_focus_apps'].astype(int) * 2) +
    (df_clean['has_digital_wellbeing_enabled'].astype(int) * 2) +
    (10 - df_clean['daily_social_media_time'].clip(0, 10)) +
    (df_clean['weekly_offline_hours'] / 4)
).round(2)

# Balance trabajo-vida (horas offline vs trabajo)
df_clean['work_life_balance'] = (df_clean['weekly_offline_hours'] / 
                                (df_clean['work_hours_per_day'] * 5)).round(2)

print(f"   Nuevas características creadas: 6")
print(f"   - productivity_gap: diferencia entre productividad percibida y real")
print(f"   - age_category: categorías de edad")
print(f"   - social_media_usage: nivel de uso de redes sociales")
print(f"   - work_intensity: intensidad de trabajo")
print(f"   - digital_wellness_score: índice de bienestar digital")
print(f"   - work_life_balance: balance trabajo-vida")

# 6. CODIFICACIÓN DE VARIABLES CATEGÓRICAS
print("\n6. PREPARANDO VARIABLES CATEGÓRICAS PARA ML...")

# One-hot encoding para variables categóricas
categorical_cols = ['gender', 'job_type', 'social_platform_preference', 
                   'age_category', 'social_media_usage', 'work_intensity']

df_encoded = pd.get_dummies(df_clean, columns=categorical_cols, prefix=categorical_cols)

print(f"   Variables categóricas codificadas: {len(categorical_cols)}")
print(f"   Nuevas columnas después de encoding: {df_encoded.shape[1]}")

# 7. ESTADÍSTICAS FINALES
print("\n=== ESTADÍSTICAS FINALES ===")
print(f"Dataset final: {df_encoded.shape}")
print(f"Variables numéricas: {len(df_encoded.select_dtypes(include=[np.number]).columns)}")
print(f"Variables booleanas: {len(df_encoded.select_dtypes(include=[bool]).columns)}")

# Verificar correlaciones altas (multicolinealidad)
numeric_cols = df_encoded.select_dtypes(include=[np.number]).columns
corr_matrix = df_encoded[numeric_cols].corr()
high_corr_pairs = []

for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.8:
            high_corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))

if high_corr_pairs:
    print(f"\nCorrelaciones altas detectadas (>0.8): {len(high_corr_pairs)}")
    for col1, col2, corr in high_corr_pairs[:5]:  # Mostrar solo las primeras 5
        print(f"   {col1} - {col2}: {corr:.3f}")
else:
    print("\nNo se detectaron correlaciones problemáticas (>0.8)")

=== INGENIERÍA DE CARACTERÍSTICAS ===

5. CREANDO NUEVAS CARACTERÍSTICAS...
   Nuevas características creadas: 6
   - productivity_gap: diferencia entre productividad percibida y real
   - age_category: categorías de edad
   - social_media_usage: nivel de uso de redes sociales
   - work_intensity: intensidad de trabajo
   - digital_wellness_score: índice de bienestar digital
   - work_life_balance: balance trabajo-vida

6. PREPARANDO VARIABLES CATEGÓRICAS PARA ML...
   Variables categóricas codificadas: 6
   Nuevas columnas después de encoding: 46

=== ESTADÍSTICAS FINALES ===
Dataset final: (30000, 46)
Variables numéricas: 17
Variables booleanas: 29

Correlaciones altas detectadas (>0.8): 3
   perceived_productivity_score - actual_productivity_score: 0.901
   actual_productivity_score - job_satisfaction_score: 0.809
   weekly_offline_hours - work_life_balance: 0.811


In [8]:
# 8. GUARDAR DATASET LIMPIO
print("=== GUARDANDO DATASET LIMPIO ===")

# Guardar el dataset completamente limpio y preparado para ML
df_encoded.to_csv('social_media_productivity_clean.csv', index=False)
print("✅ Dataset limpio guardado como 'social_media_productivity_clean.csv'")

# También guardar una versión sin encoding para análisis exploratorio
df_clean.to_csv('social_media_productivity_features.csv', index=False)
print("✅ Dataset con nuevas características guardado como 'social_media_productivity_features.csv'")

=== GUARDANDO DATASET LIMPIO ===
✅ Dataset limpio guardado como 'social_media_productivity_clean.csv'
✅ Dataset con nuevas características guardado como 'social_media_productivity_features.csv'


In [ ]:
# 9. RESUMEN FINAL DE LA LIMPIEZA
print("\n=== RESUMEN FINAL DE LA LIMPIEZA ===")
print(f"""
📊 TRANSFORMACIONES REALIZADAS:

1. VALORES NULOS:
   ✅ 13,687 valores nulos imputados con medianas
   ✅ 0 valores nulos restantes

2. OUTLIERS:
   ✅ 886 outliers tratados mediante winsorización
   ✅ Datos extremos limitados a rangos razonables

3. VALIDACIÓN DE RANGOS:
   ✅ Horas de trabajo: 0-16h
   ✅ Horas de sueño: 3-12h  
   ✅ Scores: 0-10 puntos
   ✅ Tiempo pantalla: 0-4h

4. NUEVAS CARACTERÍSTICAS:
   ✅ 6 variables derivadas creadas
   ✅ Índices de bienestar y balance trabajo-vida

5. ENCODING:
   ✅ 6 variables categóricas codificadas
   ✅ 46 columnas finales (vs 19 originales)

📈 DATASET FINAL:
   • Filas: 30,000
   • Columnas: 46
   • Variables numéricas: 17
   • Variables dummy: 29
   • Calidad: Lista para ML

⚠️  CORRELACIONES ALTAS DETECTADAS:
   • Productividad percibida vs real (0.901)
   • Productividad real vs satisfacción laboral (0.809)
   • Horas offline vs balance trabajo-vida (0.811)
   
💡 RECOMENDACIÓN: Considera eliminar una de las variables altamente correlacionadas
   para evitar multicolinealidad en modelos lineales.
""")

# Mostrar las primeras filas del dataset final
print("\n=== MUESTRA DEL DATASET FINAL ===")
print(df_encoded.head(3))
print(f"\nColumnas disponibles ({len(df_encoded.columns)}):")
for i, col in enumerate(df_encoded.columns):
    if i % 4 == 0:
        print()
    print(f"{col:<25}", end=" ")
print("\n")

=== GUARDANDO DATASET LIMPIO ===
✅ Dataset limpio guardado como 'social_media_productivity_clean.csv'
✅ Dataset con nuevas características guardado como 'social_media_productivity_features.csv'

=== RESUMEN FINAL DE LA LIMPIEZA ===

📊 TRANSFORMACIONES REALIZADAS:

1. VALORES NULOS:
   ✅ 13,687 valores nulos imputados con medianas
   ✅ 0 valores nulos restantes

2. OUTLIERS:
   ✅ 886 outliers tratados mediante winsorización
   ✅ Datos extremos limitados a rangos razonables

3. VALIDACIÓN DE RANGOS:
   ✅ Horas de trabajo: 0-16h
   ✅ Horas de sueño: 3-12h  
   ✅ Scores: 0-10 puntos
   ✅ Tiempo pantalla: 0-4h

4. NUEVAS CARACTERÍSTICAS:
   ✅ 6 variables derivadas creadas
   ✅ Índices de bienestar y balance trabajo-vida

5. ENCODING:
   ✅ 6 variables categóricas codificadas
   ✅ 46 columnas finales (vs 19 originales)

📈 DATASET FINAL:
   • Filas: 30,000
   • Columnas: 46
   • Variables numéricas: 17
   • Variables dummy: 29
   • Calidad: Lista para ML

⚠️  CORRELACIONES ALTAS DETECTADAS:
  